<a href="https://colab.research.google.com/github/ayyanarh1/tamil-nadu-school-flood-risk/blob/main/day16_giga_real_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Setup
!pip install requests pandas geopandas folium -q

import requests
import pandas as pd
import geopandas as gpd
import folium
import json

print('✅ Day 16 ready!')

✅ Day 16 ready!


In [7]:
# Realistic Tamil Nadu school dataset
# Based on real Giga data structure and Tamil Nadu geography
import pandas as pd
import numpy as np

np.random.seed(42)

# Real Tamil Nadu districts with accurate coordinates
districts = {
    'Chennai':         (13.08, 80.27, 50),
    'Cuddalore':       (11.75, 79.75, 40),
    'Nagapattinam':    (10.76, 79.84, 35),
    'Thanjavur':       (10.78, 79.13, 45),
    'Tiruchirappalli': (10.79, 78.68, 45),
    'Madurai':         (9.93,  78.12, 50),
    'Tirunelveli':     (8.71,  77.69, 40),
    'Coimbatore':      (11.01, 76.96, 50),
    'Salem':           (11.65, 78.16, 40),
    'Vellore':         (12.92, 79.13, 35),
    'Kanchipuram':     (12.83, 79.70, 30),
    'Villupuram':      (11.93, 79.49, 35),
    'Ramanathapuram':  (9.37,  78.83, 25),
    'Thoothukudi':     (8.80,  78.15, 30),
    'Puducherry':      (11.93, 79.83, 20),
}

schools = []
school_id = 1

for district, (lat, lon, count) in districts.items():
    for i in range(count):
        # Add small random offset for each school
        s_lat = lat + np.random.uniform(-0.3, 0.3)
        s_lon = lon + np.random.uniform(-0.3, 0.3)

        # School type
        school_type = np.random.choice(
            ['Primary', 'Upper Primary', 'Secondary', 'Higher Secondary'],
            p=[0.4, 0.3, 0.2, 0.1]
        )

        # Connectivity — coastal districts have less
        coastal = district in [
            'Chennai', 'Cuddalore', 'Nagapattinam',
            'Puducherry', 'Ramanathapuram', 'Thoothukudi'
        ]
        conn_prob = 0.3 if coastal else 0.6
        connectivity = np.random.choice(
            ['connected', 'not_connected'],
            p=[conn_prob, 1-conn_prob]
        )

        schools.append({
            'school_id': f'IND_{school_id:05d}',
            'school_name': f'{school_type} School {district} {i+1}',
            'district': district,
            'state': 'Tamil Nadu',
            'latitude': round(s_lat, 4),
            'longitude': round(s_lon, 4),
            'connectivity_status': connectivity,
            'school_type': school_type,
            'country_code': 'IND'
        })
        school_id += 1

giga_df = pd.DataFrame(schools)

print(f'✅ Tamil Nadu Giga dataset created!')
print(f'Total schools: {len(giga_df)}')
print(f'Districts: {giga_df.district.nunique()}')
print(f'Connected: {len(giga_df[giga_df.connectivity_status == "connected"])}')
print(f'Not connected: {len(giga_df[giga_df.connectivity_status == "not_connected"])}')
print()
print(giga_df.groupby("district")["school_id"].count())

✅ Tamil Nadu Giga dataset created!
Total schools: 570
Districts: 15
Connected: 266
Not connected: 304

district
Chennai            50
Coimbatore         50
Cuddalore          40
Kanchipuram        30
Madurai            50
Nagapattinam       35
Puducherry         20
Ramanathapuram     25
Salem              40
Thanjavur          45
Thoothukudi        30
Tiruchirappalli    45
Tirunelveli        40
Vellore            35
Villupuram         35
Name: school_id, dtype: int64


In [10]:
# Extract flood risk for all 570 schools
import ee
import pandas as pd
import numpy as np

ee.Authenticate()
ee.Initialize(project='tamil-nadu-flood-risk')


print('Extracting flood risk for 570 schools...')
print('This may take 5-10 minutes...')

# Tamil Nadu boundary
tamil_nadu = ee.Geometry.Rectangle([76.0, 8.0, 80.5, 13.5])

# Load datasets
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater') \
    .select('occurrence') \
    .clip(tamil_nadu)

gfd = ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1') \
    .select('flooded').sum().clip(tamil_nadu)

s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains(
        'transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .filterBounds(tamil_nadu) \
    .filterDate('2023-10-01', '2023-12-31') \
    .select('VV')

s1_median = s1.median().clip(tamil_nadu)
flood_mask = s1_median.lt(-15).rename('flood_pixel')

# Process in batches of 50
batch_size = 50
all_results = []

for batch_start in range(0, len(giga_df), batch_size):
    batch = giga_df.iloc[batch_start:batch_start + batch_size]
    batch_num = batch_start // batch_size + 1
    total_batches = len(giga_df) // batch_size + 1
    print(f'Processing batch {batch_num}/{total_batches}...')

    # Build features
    features = []
    for _, row in batch.iterrows():
        point = ee.Geometry.Point([row['longitude'], row['latitude']])
        buffer = point.buffer(500)
        features.append(ee.Feature(buffer, {
            'school_id':   row['school_id'],
            'school_name': row['school_name'],
            'district':    row['district'],
            'connectivity': row['connectivity_status']
        }))

    schools_ee = ee.FeatureCollection(features)

    try:
        # Extract JRC
        jrc_stats = jrc.reduceRegions(
            collection=schools_ee,
            reducer=ee.Reducer.mean(),
            scale=30
        )
        jrc_data = [
            f['properties']
            for f in jrc_stats.getInfo()['features']
        ]
        jrc_batch = pd.DataFrame(jrc_data)
        jrc_batch = jrc_batch.rename(
            columns={'mean': 'water_occurrence_pct'}
        )
        jrc_batch['water_occurrence_pct'] = \
            jrc_batch['water_occurrence_pct'].fillna(0).round(2)

        # Extract GFD
        gfd_stats = gfd.reduceRegions(
            collection=schools_ee,
            reducer=ee.Reducer.mean(),
            scale=250
        )
        gfd_data = [
            f['properties']
            for f in gfd_stats.getInfo()['features']
        ]
        gfd_batch = pd.DataFrame(gfd_data)
        gfd_batch = gfd_batch.rename(
            columns={'mean': 'flood_frequency'}
        )
        gfd_batch['flood_frequency'] = \
            gfd_batch['flood_frequency'].fillna(0).round(2)

        # Merge batch
        batch_result = jrc_batch[[
            'school_id', 'school_name',
            'district', 'connectivity',
            'water_occurrence_pct'
        ]].merge(
            gfd_batch[['school_id', 'flood_frequency']],
            on='school_id'
        )
        all_results.append(batch_result)

    except Exception as e:
        print(f'  ⚠ Batch {batch_num} error: {str(e)[:50]}')
        continue

# Combine all batches
results_df = pd.concat(all_results, ignore_index=True)
print(f'\n✅ Extracted risk for {len(results_df)} schools!')
print(results_df.head())

Extracting flood risk for 570 schools...
This may take 5-10 minutes...
Processing batch 1/12...
Processing batch 2/12...
Processing batch 3/12...
Processing batch 4/12...
Processing batch 5/12...
Processing batch 6/12...
Processing batch 7/12...
Processing batch 8/12...
Processing batch 9/12...
Processing batch 10/12...
Processing batch 11/12...
Processing batch 12/12...

✅ Extracted risk for 570 schools!
   school_id                     school_name district   connectivity  \
0  IND_00001      Secondary School Chennai 1  Chennai  not_connected   
1  IND_00002        Primary School Chennai 2  Chennai  not_connected   
2  IND_00003        Primary School Chennai 3  Chennai  not_connected   
3  IND_00004        Primary School Chennai 4  Chennai      connected   
4  IND_00005  Upper Primary School Chennai 5  Chennai      connected   

   water_occurrence_pct  flood_frequency  
0                  0.00             0.00  
1                 11.73             0.24  
2                 99.24      

In [11]:
# Build risk scores for all 570 schools
import pandas as pd
import numpy as np

# Coastal districts
coastal_districts = [
    'Chennai', 'Cuddalore', 'Nagapattinam',
    'Puducherry', 'Ramanathapuram', 'Thoothukudi'
]

# Normalise scores
results_df['jrc_norm'] = results_df['water_occurrence_pct'].round(1)
results_df['gfd_norm'] = (
    results_df['flood_frequency'] /
    max(results_df['flood_frequency'].max(), 1) * 100
).round(1)

# Flood score
results_df['flood_score'] = (
    0.60 * results_df['jrc_norm'] +
    0.40 * results_df['gfd_norm']
).round(1)

# Connectivity penalty
results_df['conn_penalty'] = results_df['connectivity'].apply(
    lambda x: 15 if x == 'not_connected' else 0
)

# Exposure score
results_df['exposure'] = results_df['district'].apply(
    lambda x: 85 if x in coastal_districts else 45
)

# Final score
results_df['final_score'] = (
    results_df['flood_score'] +
    results_df['conn_penalty']
).round(1)

# Risk tier
def risk_tier(score):
    if score >= 50:   return 'CRITICAL'
    elif score >= 25: return 'HIGH'
    elif score >= 10: return 'MEDIUM'
    else:             return 'LOW'

results_df['risk_tier'] = results_df['final_score'].apply(risk_tier)

# Summary
print('=== 570 SCHOOL RISK SUMMARY ===\n')
print('Risk distribution:')
print(results_df['risk_tier'].value_counts())
print()
print('By district:')
district_summary = results_df.groupby('district').agg(
    schools=('school_id', 'count'),
    critical=('risk_tier', lambda x: (x == 'CRITICAL').sum()),
    avg_score=('final_score', 'mean')
).round(1)
district_summary = district_summary.sort_values(
    'avg_score', ascending=False
)
print(district_summary)

=== 570 SCHOOL RISK SUMMARY ===

Risk distribution:
risk_tier
MEDIUM      236
LOW         224
CRITICAL     83
HIGH         27
Name: count, dtype: int64

By district:
                 schools  critical  avg_score
district                                     
Ramanathapuram        25        14       44.7
Thoothukudi           30        15       43.1
Cuddalore             40        15       37.6
Nagapattinam          35        13       35.9
Chennai               50        17       34.6
Puducherry            20         7       32.7
Kanchipuram           30         1       13.4
Thanjavur             45         0       12.6
Tiruchirappalli       45         0       12.2
Villupuram            35         0       10.0
Vellore               35         0        9.1
Tirunelveli           40         0        8.6
Coimbatore            50         1        7.8
Salem                 40         0        7.3
Madurai               50         0        7.1


In [12]:
# Large scale risk map for 570 schools
import folium
from folium.plugins import MarkerCluster
import pandas as pd

# Merge with original giga_df for coordinates
full_df = results_df.merge(
    giga_df[['school_id', 'latitude', 'longitude', 'school_type']],
    on='school_id'
)

# Build map
m = folium.Map(
    location=[10.5, 78.5],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Use MarkerCluster for 570 schools
cluster = MarkerCluster(name='All schools').add_to(m)

color_map = {
    'CRITICAL': '#CC0000',
    'HIGH':     '#FF6600',
    'MEDIUM':   '#FFAA00',
    'LOW':      '#2D8A4E'
}

for _, row in full_df.iterrows():
    color = color_map.get(row['risk_tier'], 'gray')
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        tooltip=f"{row['school_name']} — {row['risk_tier']}",
        popup=folium.Popup(
            f"<b>{row['school_name']}</b><br>"
            f"District: {row['district']}<br>"
            f"Type: {row['school_type']}<br>"
            f"Connectivity: {row['connectivity']}<br>"
            f"Flood score: {row['flood_score']}<br>"
            f"Risk: <b>{row['risk_tier']}</b>",
            max_width=250
        )
    ).add_to(cluster)

# Legend
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;
     background:white;padding:14px;border-radius:10px;
     border:1px solid #ccc;font-size:12px;z-index:1000;">
     <b>570 Tamil Nadu Schools</b><br>
     <b>Flood Risk Assessment</b><br><br>
     <span style='color:#CC0000;font-size:16px'>●</span> Critical<br>
     <span style='color:#FF6600;font-size:16px'>●</span> High<br>
     <span style='color:#FFAA00;font-size:16px'>●</span> Medium<br>
     <span style='color:#2D8A4E;font-size:16px'>●</span> Low<br><br>
     <i>Click clusters to zoom in</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(m)
m.save('tamil_nadu_570_schools_map.html')
print('✅ 570 school map saved!')
display(m)

✅ 570 school map saved!


In [13]:
# Save all outputs
from google.colab import files

# Save full dataset
full_df.to_csv('tamil_nadu_570_schools_risk.csv', index=False)
print('✅ CSV saved!')

# Download
files.download('tamil_nadu_570_schools_map.html')
files.download('tamil_nadu_570_schools_risk.csv')
print('✅ All files downloaded!')

print()
print('=== DAY 16 SUMMARY ===')
print(f'Total schools analysed: {len(full_df)}')
print(f'Districts covered: {full_df.district.nunique()}')
print(f'CRITICAL schools: {len(full_df[full_df.risk_tier == "CRITICAL"])}')
print(f'HIGH schools: {len(full_df[full_df.risk_tier == "HIGH"])}')
print(f'MEDIUM schools: {len(full_df[full_df.risk_tier == "MEDIUM"])}')
print(f'LOW schools: {len(full_df[full_df.risk_tier == "LOW"])}')
print()
print('Most at-risk district: Ramanathapuram')
print('Safest district: Madurai')
print()
print('Scale comparison:')
print('  Day 1-7:  15 sample schools')
print('  Day 16:   570 Giga-style schools')
print('  Full Giga: 5,000+ schools (ready to scale!)')

✅ CSV saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files downloaded!

=== DAY 16 SUMMARY ===
Total schools analysed: 570
Districts covered: 15
CRITICAL schools: 83
HIGH schools: 27
MEDIUM schools: 236
LOW schools: 224

Most at-risk district: Ramanathapuram
Safest district: Madurai

Scale comparison:
  Day 1-7:  15 sample schools
  Day 16:   570 Giga-style schools
  Full Giga: 5,000+ schools (ready to scale!)
